<div align="center">
    <span style="font-size:30px">
        <strong>
            <!-- Símbolo de Python -->
            <img
                src="https://cdn3.emoji.gg/emojis/1887_python.png"
                style="margin-bottom:-5px"
                width="30px" 
                height="30px"
            >
            <!-- Título -->
            Python para Geólogos
            <!-- Versión -->
            <img 
                src="https://img.shields.io/github/release/kevinalexandr19/manual-python-geologia.svg?style=flat&label=&color=blue"
                style="margin-bottom:-2px" 
                width="40px"
            >
        </strong>
    </span>
    <br>
    <span>
        <!-- Github del proyecto -->
        <a href="https://github.com/kevinalexandr19/manual-python-geologia" target="_blank">
            <img src="https://img.shields.io/github/stars/kevinalexandr19/manual-python-geologia.svg?style=social&label=Github Repo">
        </a>
        &nbsp;&nbsp;
        <!-- Licencia -->
        <img src="https://img.shields.io/github/license/kevinalexandr19/manual-python-geologia.svg?color=forestgreen">
        &nbsp;&nbsp;
        <!-- Release date -->
        <img src="https://img.shields.io/github/release-date/kevinalexandr19/manual-python-geologia?color=gold">
    </span>
    <br>
    <span>
        <!-- Perfil de LinkedIn -->
        <a target="_blank" href="https://www.linkedin.com/in/kevin-alexander-gomez/">
            <img src="https://img.shields.io/badge/-Kevin Alexander Gomez-5eba00?style=social&logo=linkedin">
        </a>
        &nbsp;&nbsp;
        <!-- Perfil de Github -->
        <a target="_blank" href="https://github.com/kevinalexandr19">
            <img src="https://img.shields.io/github/followers/kevinalexandr19.svg?style=social&label=kevinalexandr19&maxAge=2592000">
        </a>
    </span>
    <br>
</div>

***

<span style="color:lightgreen; font-size:25px">**PG301 - Geoestadística** </span>

Bienvenido al curso!!!

Vamos a revisar las bases de la <span style="color:gold">geoestadística</span> usando ejemplos en Python. <br>
Es necesario que tengas un conocimiento previo en programación con Python, estadística y geología general.

<span style="color:gold; font-size:20px">**Variograma 1D**</span>

***
- [¿Qué es gamma?](#parte-1)
- [Procesamiento de datos](#parte-2)
- [Creación del variograma experimental](#parte-3)
- [Intervalos de espaciamiento](#parte-4)
- [Ajuste del modelo de variograma](#parte-5)

***

In [ ]:
import sys

if sys.platform == "emscripten":
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'scipy', 'ipywidgets'])

<a id="parte-1"></a>

### <span style="color:lightgreen">**¿Qué es gamma?**</span>
***

<span style="color:gold">**Gamma**</span> es el módulo del curso para explorar variogramas en Jupyter. La clase `Variogram` reúne valores y coordenadas, calcula pares, los agrupa por distancia y permite ajustar un modelo. Podemos seguir los mismos pasos con código o con los controles interactivos.

Aquí analizaremos cobre a lo largo de un solo sondaje. La perspectiva es similar a PG200: definimos qué representa una fila, qué entra en el cálculo, qué sale y qué evidencia necesitaríamos para usar ese modelo en otro contexto.

<a id="parte-2"></a>

### <span style="color:lightgreen">**Procesamiento de datos**</span>
***

En `assay.csv` cada fila representa un tramo, con profundidad inicial `FROM`, profundidad final `TO` y ley de cobre `CU_pct`. Seleccionamos DH010 y usamos la profundidad media del tramo como su coordenada.

Esta distancia se mide **a lo largo del sondaje**. Para obtener distancias 3D entre muestras harían falta las coordenadas calculadas con la trayectoria del taladro. El valor representa un intervalo, por lo que longitudes diferentes pueden cambiar la variabilidad. Revisamos ese soporte antes de calcular.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from files.gamma import Variogram

tramos = pd.read_csv("files/assay.csv", usecols=["ID", "FROM", "TO", "CU_pct"])
data = tramos.loc[tramos["ID"] == "DH010", ["FROM", "TO", "CU_pct"]].copy()
validos = np.isfinite(data).all(axis=1) & (data["TO"] > data["FROM"])
print(f"Tramos utilizables: {validos.sum()} de {len(data)}")
data = data.loc[validos].sort_values("FROM").reset_index(drop=True)
data["Longitud_m"] = data["TO"] - data["FROM"]
data["coords"] = (data["FROM"] + data["TO"]) / 2
print("Longitudes de tramo (m):")
print(data["Longitud_m"].round(2).value_counts().sort_index())
data.head()

In [ ]:
values = data["CU_pct"].to_numpy()
coords = data["coords"].to_numpy()

fig, ax = plt.subplots(figsize=(9, 3.1))
ax.plot(coords, values, "-", lw=.85, color="#0072B2")
ax.scatter(coords, values, s=12, color="#0072B2")
ax.set(title="Sondaje DH010 · perfil de cobre", xlabel="Profundidad a lo largo del sondaje (m)",
       ylabel="Cu (%)")
ax.grid(alpha=.2)
fig.tight_layout()
plt.show()

La mayoría de tramos mide 2 m y hay uno ligeramente más largo. En esta práctica lo conservamos y verificamos su longitud. En un estudio de estimación podríamos componer a un soporte común y revisar contactos, cambios en recuperación y posibles tendencias en profundidad.

<a id="parte-3"></a>

### <span style="color:lightgreen">**Creación del variograma experimental**</span>
***

En [Introducción al Variograma](pg301a_variograma.ipynb) calculamos un par a mano. Vamos a repetirlo con los tramos 0 y 1 de DH010 y localizar las mismas filas en el resultado del módulo. A partir de 305 tramos hay 46 360 combinaciones posibles. Aquí conservaremos pares hasta 240 m.

In [ ]:
distancia_manual = abs(coords[1] - coords[0])
semivarianza_manual = (values[1] - values[0]) ** 2 / 2
print(f"Par 0–1: {distancia_manual:g} m | aporte {semivarianza_manual:.4f} (% Cu)²")

var1d = Variogram(values=values, coords=coords, lagDist=4, max_lag=240)
print(var1d)
pares = var1d.pair_table()
pares.loc[(pares["i"] == 0) & (pares["j"] == 1)]

In [ ]:
clases = var1d.lag_table()
clases.head().round(4)

`h` es el centro nominal de una clase. `h_media` es donde se sitúa el punto en la figura. La primera clase llega hasta 6 m e incluye separaciones positivas de 2, 4 y 6 m. Comprueba que promediar `semivarianza_par` de la clase produce el primer valor de `gamma`.

<a id="parte-4"></a>

### <span style="color:lightgreen">**Intervalos de espaciamiento (binning)**</span>
***

Probemos 2, 4 y 8 m. Un paso pequeño muestra cambios con mayor detalle y puede dejar clases con poca información. Al ampliar el paso juntamos más pares y podemos suavizar estructuras. Usaremos 4 m para el primer ajuste, como en el ejemplo original.

In [ ]:
fig, (ax, conteos) = plt.subplots(2, 1, figsize=(9, 5), sharex=True,
                                 gridspec_kw={"height_ratios": [2.5, 1]})
alternativa = Variogram(values, coords, lagDist=2, max_lag=240)
for paso, color in [(2, "#526575"), (4, "#0072B2"), (8, "#B84D12")]:
    alternativa.updateLagDist(paso)
    ax.plot(alternativa.mean_lag, alternativa.gamma, ".-", label=f"Paso {paso} m",
            color=color, alpha=.85)
    conteos.plot(alternativa.mean_lag, alternativa.pairs, ".-", color=color)
ax.set(ylabel="Semivarianza (% Cu)²", title="Mismos pares, distintas clases")
ax.legend(frameon=False, ncol=3)
conteos.set(xlabel="Distancia media del par (m)", ylabel="Pares")
for eje in (ax, conteos):
    eje.grid(alpha=.2)
fig.tight_layout()
plt.show()

**Exploración opcional.** `Aplicar paso` vuelve a agrupar los mismos pares de una copia del ejemplo. Intenta 6 m y compara los límites. La sección siguiente usa el objeto `var1d` con el paso de 4 m para que la ejecución sea reproducible.

In [ ]:
explorar_paso = Variogram(values, coords, lagDist=4, max_lag=240)
control_paso = explorar_paso.selectLagDist()

<a id="parte-5"></a>

### <span style="color:lightgreen">**Ajuste del modelo de variograma**</span>
***

Vamos a comparar modelos sobre las mismas clases, hasta una distancia media de 200 m y con al menos 30 pares. Dependemos de este perfil y de esa ventana. Gamma pondera el ajuste por el número de pares y usa la distancia media de cada clase. Como los pares pueden compartir muestras, usamos el conteo como una ponderación descriptiva.

Compara qué tan rápido se aproxima cada modelo a la meseta. El modelo lineal se muestra como referencia sin meseta ni alcance. El alcance de exponencial y gaussiano se expresa al 95% de la contribución de la estructura.

In [ ]:
min_pares = 30
ventana_m = 200
n_clases = int(((clases["h_media"] <= ventana_m) & (clases["pares"] >= min_pares)).sum())
ajustes = [var1d.fit(m, n_points=n_clases, min_pairs=min_pares)
           for m in ["linear", "exponential", "gaussian", "spherical"]]
columnas = ["model", "n_points", "c0", "c", "sill", "a", "mse", "r2"]
pd.DataFrame(ajustes)[columnas].round(4)

Miramos el esférico para conectar con el ejemplo de la pepita. Los puntos rellenos fueron usados para ajustar. La línea discontinua muestra la curva fuera de esa ventana. R² y MSE describen el ajuste a **las clases usadas**, con el mismo peso por clase al reportar la métrica. Antes de usar el modelo para estimar leyes en otros sondajes, habría que verificar direcciones, dominios y desempeño al reservar sondajes completos.

In [ ]:
modelo_base = var1d.model  # El último ajuste de la lista es esférico.
fig, ejes = var1d.plot(distance_unit="m", value_unit="% Cu")
plt.show()
print(f"Pepita: {modelo_base['c0']:.3f} | Meseta: {modelo_base['sill']:.3f} | "
      f"Alcance: {modelo_base['a']:.1f} m")

**Exploración opcional.** En el panel puedes elegir la forma y el número de clases. Desactiva **Autoajuste** para escribir tus propios valores y usa **Aplicar modelo** para ver el resultado. En una recta, `C/a` es la pendiente. En los demás, `a` es el alcance reportado. El panel trabaja sobre una copia del ejemplo.

In [ ]:
explorar_modelo = Variogram(values, coords, lagDist=4, max_lag=240)
explorar_modelo.fit("spherical", n_points=n_clases, min_pairs=min_pares)
control_modelo = explorar_modelo.fitting(distance_unit="m", value_unit="% Cu")

El diccionario `model` tiene valores que podemos serializar. Conservamos el intervalo, el corte y la unidad con el ajuste para poder repetirlo. Para evaluar el modelo fuera del objeto, `popt` guarda los parámetros internos de la función, mientras que `a` informa el alcance práctico cuando corresponde.

In [ ]:
resultado = {
    "sondaje": "DH010",
    "variable": "CU_pct",
    "unidad_distancia": "m a lo largo del sondaje",
    "paso_m": var1d.lagDist,
    "max_lag_m": var1d.max_lag,
    "ventana_ajuste_m": ventana_m,
    **modelo_base,
}
resultado_json = json.dumps(resultado, ensure_ascii=False, indent=2, allow_nan=False)
print(resultado_json)
# Para descargar una copia desde JupyterLite, guarda el JSON desde esta celda.
exportar = False
if exportar:
    Path("variograma_DH010.json").write_text(resultado_json, encoding="utf8")

**Comprueba tu lectura.** Repite el ajuste con un paso de 8 m y un corte de 150 m. ¿Qué filas cambian en la tabla de clases? ¿Por qué podría cambiar el alcance? Si usas IA para proponer otro ajuste, verifica primero la unidad, el soporte, los límites de clase y el rango de distancias que realmente tienen pares.

El siguiente notebook estudia otro efecto del diseño de muestreo: [desagrupamiento por celdas](pg301c_declustering.ipynb).